# Chroma — basics

Local Chroma experiments. Persist under `../data` so files stay in this engine folder.

```python
# !pip install chromadb --user
import chromadb
import pandas as pd
client = chromadb.PersistentClient(path="../data")
```


In [25]:
import chromadb
import pandas as pd
client = chromadb.PersistentClient(path="../data")

In [26]:
# Dataset: https://www.kaggle.com/datasets/nulldata/medium-post-titles
# File is chroma/resource/medium_post_titles.csv (not /chroma/resources/...).
# This notebook lives in chroma/notebooks/, so go one level up.
from pathlib import Path
import pandas as pd

csv_path = Path("..") / "resource" / "medium_post_titles.csv"
df = pd.read_csv(csv_path)

df = df.dropna()
df = df[~df["subtitle_truncated_flag"]]

topics_of_interest = ["artificial-intelligence", "data-science", "machine-learning"]

df = df[df["category"].isin(topics_of_interest)]

df["text"] = df["title"] + df["subtitle"]

df["meta"] = df.apply(
    lambda x: {"text": x["text"], "category": x["category"]},
    axis=1,
)


In [27]:
df.head(2)

,category,title,subtitle,subtitle_truncated_flag,text,meta
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we...","{'text': '""Can I Train my Model on Your Comput..."
289,data-science,(Robot) data scientists as a service,Automating data science with symbolic regressi...,False,(Robot) data scientists as a serviceAutomating...,{'text': '(Robot) data scientists as a service...


## Chroma DB setup

Chroma 1.x dropped `chroma_db_impl="duckdb+parquet"`. Use `PersistentClient` (already created in the import cell) and `get_or_create_collection` so re-runs do not fail.

In [28]:
# Not needed on chromadb 1.x:
# from chromadb.config import Settings
# chromadb.Client(Settings(persist_directory=..., chroma_db_impl="duckdb+parquet"))
print("client path: ../data")
print(client)

client path: ../data


In [29]:
# Persistent store is chroma/data (see PersistentClient in the import cell).
# get_or_create_collection: safe if you re-run this cell.
article_collection = client.get_or_create_collection(name="medium-article")
article_collection.count()

4082

Default embedding is Chroma’s built-in function (all-MiniLM-L6-v2) unless you pass `embedding_function=`. Data is written under `chroma/data`, not `medium-chroma-db`.

## Data Insertion

In [30]:
# inserting data

article_collection.upsert(
    ids=[f"{x}" for x in df.index.tolist()],
    documents=df['text'].tolist(),
    metadatas=df['meta'].tolist()    
)

In [31]:
article_collection.count()

4082

In [32]:
qry_str = "best data science library?"

In [33]:
article_collection.query(query_texts=qry_str, n_results=1)

{'ids': [['65427']],
 'embeddings': None,
 'documents': [['My Favorite Data Science/Machine Learning ResourcesA summary of sources to get into Data Science']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'data-science',
    'text': 'My Favorite Data Science/Machine Learning ResourcesA summary of sources to get into Data Science'}]],
 'distances': [[0.6087141633033752]]}

In [34]:
# article_collection.delete()

In [36]:
# Find the exact database location

from pathlib import Path

print(Path("../data").resolve())



/Users/sushilpratapsingh/projects/vector db/chroma/data
